# 🔍 Floor Plan Analysis: Multi-Format Inference & Evaluation
### Parse JPG, PNG, and SVG Blueprints with Fine-Tuned Qwen3-VL 8B

This notebook demonstrates:
1. Loading the fine-tuned LoRA adapter from Google Drive.
2. Analyzing floor plans in **JPG, PNG, and SVG vector formats**.
3. Extracting structured metadata: **rooms, length, width, doors, windows, and normalized coordinates**.
4. Visualizing detected bounding boxes and dimensions overlaid on the original blueprints.
5. Evaluating Mean Intersection-over-Union (mIoU) and detection accuracy on your validation dataset.

In [ ]:
# Step 1: Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/floorplan_reader_project'
except Exception:
    PROJECT_DIR = './data/colab_run'

LORA_MODEL_DIR = f'{PROJECT_DIR}/models/qwen3_vl_floorplan_lora'
print(f"Loading LoRA weights from: {LORA_MODEL_DIR}")

In [ ]:
# Step 2: Install dependencies
!pip install -q unsloth trl peft bitsandbytes cairosvg svglib reportlab pydantic matplotlib

In [ ]:
# Step 3: Load predictor
import sys
if '.' not in sys.path:
    sys.path.append('.')

import os
from floorplan_reader.inference.predictor import FloorPlanPredictor
from floorplan_reader.visualization.visualizer import visualize_floorplan
from floorplan_reader.dataset.synthetic_adapter import generate_mock_synthetic_sample
import matplotlib.pyplot as plt
from PIL import Image

if os.path.exists(LORA_MODEL_DIR):
    print("Loading fine-tuned LoRA model for inference...")
    predictor = FloorPlanPredictor.from_pretrained_lora(
        base_model_id="unsloth/Qwen3-VL-8B-Instruct-unsloth-bnb-4bit",
        lora_weights_path=LORA_MODEL_DIR,
        load_in_4bit=True,
    )
else:
    print("LoRA directory not found. Initializing in demonstration mock mode...")
    # Initialize in mock mode so this notebook can be tested right away
    import json
    demo_json = json.dumps({
        "rooms": [
            {"id": "room_1", "name": "bedroom", "box_2d": [80, 60, 480, 460], "detected_label_text": "BEDROOM 3.8m x 3.6m"},
            {"id": "room_2", "name": "kitchen", "box_2d": [80, 520, 450, 920], "detected_label_text": "KITCHEN 3.2m x 4.0m"},
            {"id": "room_3", "name": "living_room", "box_2d": [500, 60, 920, 520], "detected_label_text": "LIVING ROOM 4.8m x 4.5m"},
            {"id": "room_4", "name": "bathroom", "box_2d": [500, 550, 920, 920], "detected_label_text": "BATHROOM 2.4m x 3.2m"}
        ],
        "doors": [
            {"id": "door_1", "type": "single_swing", "box_2d": [470, 180, 510, 230]},
            {"id": "door_2", "type": "single_swing", "box_2d": [470, 650, 510, 700]}
        ],
        "windows": [
            {"id": "win_1", "type": "standard", "box_2d": [75, 160, 90, 320], "wall_side": "north"},
            {"id": "win_2", "type": "standard", "box_2d": [75, 620, 90, 800], "wall_side": "north"}
        ]
    })
    predictor = FloorPlanPredictor(
        mock_generator_fn=lambda img, prompt: f"```json\n{demo_json}\n```"
    )
print("Predictor ready!")

### Step 4: Run Inference on Any Floor Plan (JPG, PNG, or SVG)

In [ ]:
# Generate or specify an SVG floor plan
sample_svg_path, _ = generate_mock_synthetic_sample('./test_samples', 'demo_blueprint', format='svg')

# Run prediction
analysis = predictor.predict(sample_svg_path, pixels_per_meter=50.0)

# Visualize overlay
viz_img = visualize_floorplan(sample_svg_path, analysis)

# Plot side-by-side comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 8))
from floorplan_reader.converters.image_preprocessor import load_and_preprocess_floorplan
raw_img, _, _ = load_and_preprocess_floorplan(sample_svg_path)

axes[0].imshow(raw_img)
axes[0].set_title("Original Floor Plan (Vector SVG Rasterized)", fontsize=14)
axes[0].axis("off")

axes[1].imshow(viz_img)
axes[1].set_title(f"Detected Layout ({analysis.metadata.total_rooms} Rooms, {analysis.metadata.total_doors} Doors, {analysis.metadata.total_windows} Windows)", fontsize=14)
axes[1].axis("off")
plt.tight_layout()
plt.show()

### Step 5: Inspect Structured JSON Output

In [ ]:
import json
print(json.dumps(analysis.to_clean_dict(), indent=2))

### Step 6: Quantitative Validation Set Evaluation (mIoU and Precision/Recall)

In [ ]:
from floorplan_reader.schema import BoundingBox2D

def evaluate_sample(ground_truth_dict, predicted_analysis, iou_threshold=0.5):
    gt_rooms = ground_truth_dict.get("rooms", [])
    pred_rooms = predicted_analysis.rooms
    
    matched_gt = set()
    ious = []
    
    for pr in pred_rooms:
        best_iou = 0.0
        best_idx = -1
        for idx, gt in enumerate(gt_rooms):
            if idx in matched_gt:
                continue
            gt_box = BoundingBox2D.parse_from_list_or_dict(gt["box_2d"])
            score = pr.box_2d.iou(gt_box)
            if score > best_iou:
                best_iou = score
                best_idx = idx
        if best_iou >= iou_threshold:
            matched_gt.add(best_idx)
            ious.append(best_iou)
            
    precision = len(matched_gt) / max(1, len(pred_rooms))
    recall = len(matched_gt) / max(1, len(gt_rooms))
    mean_iou = sum(ious) / max(1, len(ious))
    return {"precision": precision, "recall": recall, "mean_iou": mean_iou}

print("Evaluation helper defined. Call evaluate_sample(gt_json, predicted_analysis) across your validation split!")